# EDA v2: Flow-Based Gentrification in London
**Strategy: Fixed 2010 IMD Baseline + IMD-Change Validation (2021 Census O-D only)**

### Methodology
- **Cascading Flow Index**: Uses **IMD 2010** (fixed baseline) to assign wealth deciles to
  MSOAs, then measures cascade flows from the **2021 Census O-D** data.
- **Temporal Validation**: Independently compute IMD change (2010 → 2019) per MSOA.
  If the cascade index correctly identifies gentrifying areas, those MSOAs
  should also show declining deprivation over the preceding decade.
- **Geography**: All analysis harmonised to **2011 MSOA codes** using the ONS
  MSOA 2011-to-2021 lookup. MSOAs that were split/merged are excluded.

---

### Structure
- **Part A (Sections 1–5)**: Data loading, preprocessing & aggregation — no plots
- **Part B (Sections 6–11)**: Analysis, visualisation & export

---
# PART A — DATA LOADING & PREPROCESSING
---

## 1. Setup & Load All Datasets

| File | Description |
|------|-------------|
| `ODMG01EW_MSOA.csv` | 2021 Census migration O-D (MSOA level) |
| `imd_2019.csv` | IMD 2019 at LSOA level |
| `imd_2010.xls` | IMD 2010 at LSOA level |
| `NSPCL_NOV22_UK_LU.csv` | Postcode lookup (2011 geographies) |
| `msoa_2011_to_2021_lookup.csv` | MSOA 2011 ↔ 2021 code mapping |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pyprojroot import here
from scipy import stats

sns.set_theme(style='whitegrid', font_scale=1.1)

ROOT = here()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# ---- File paths ----
imd_2010_path       = DATA_DIR / 'imd_2010.xls'
imd_2019_path       = DATA_DIR / 'imd_2019.csv'
census_od_2021_path = DATA_DIR / 'ODMG01EW_MSOA.csv'
lookup_path         = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'
msoa_lookup_path    = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'

In [ ]:
# ---- Load IMD ----
imd_2019 = pd.read_csv(imd_2019_path)
imd_2019.columns = imd_2019.columns.str.strip()

imd_2010 = pd.read_excel(imd_2010_path, sheet_name='IMD 2010')
imd_2010.columns = imd_2010.columns.str.strip()

# ---- Load Census O-D ----
census_od_2021 = pd.read_csv(census_od_2021_path)

# ---- Load Lookups ----
lookup = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
msoa_11_21 = pd.read_csv(msoa_lookup_path)

# Quick overview
for name, df in [('IMD 2010', imd_2010), ('IMD 2019', imd_2019),
                  ('Census O-D 2021', census_od_2021), ('MSOA Lookup', msoa_11_21)]:
    print(f'\n===== {name} =====')
    print(f'Shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
    display(df.head(2))

---
## 2. Geography Harmonisation: 2021 MSOA Codes → 2011

The 2021 census uses 2021 MSOA codes, but IMD and our lookup use 2011 codes.
Most MSOAs are unchanged (~97.5%). We keep only those with a clean 1:1 mapping.

In [ ]:
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()

# Keep only unchanged MSOAs (same code in 2011 and 2021)
unchanged = msoa_11_21[msoa_11_21['msoa11cd'] == msoa_11_21['msoa21cd']].copy()
msoa21_to_11 = dict(zip(unchanged['msoa21cd'], unchanged['msoa11cd']))

print(f'Total MSOAs in lookup: {len(msoa_11_21)}')
print(f'Unchanged (usable):    {len(unchanged)}')
print(f'Dropped (split/merged): {len(msoa_11_21) - len(unchanged)}')

In [ ]:
# Map 2021 O-D endpoints to 2011 MSOA codes
ORIGIN_COL = 'Migrant MSOA one year ago code'
DEST_COL   = 'Middle layer Super Output Areas code'

census_od_2021['origin_msoa11'] = census_od_2021[ORIGIN_COL].map(msoa21_to_11)
census_od_2021['dest_msoa11']   = census_od_2021[DEST_COL].map(msoa21_to_11)

n_total = len(census_od_2021)
n_mapped = census_od_2021[['origin_msoa11', 'dest_msoa11']].notna().all(axis=1).sum()
print(f'2021 O-D records: {n_total:,}')
print(f'Both endpoints mapped to 2011 codes: {n_mapped:,} ({n_mapped/n_total*100:.1f}%)')

---
## 3. London Filter & IMD Aggregation to MSOA

In [ ]:
london_boroughs = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent',
    'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich',
    'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering',
    'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea',
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham',
    'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton',
    'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]

london_lookup = (
    lookup[lookup['ladnm'].isin(london_boroughs)]
    [['lsoa11cd', 'msoa11cd', 'ladnm']]
    .drop_duplicates()
)

print(f'London LSOAs: {london_lookup["lsoa11cd"].nunique()}')
print(f'London MSOAs: {london_lookup["msoa11cd"].nunique()}')
print(f'Boroughs:     {london_lookup["ladnm"].nunique()}')

In [ ]:
# ---- IMD 2010 → MSOA ----
IMD_2010_LSOA_COL  = 'LSOA CODE'
IMD_2010_SCORE_COL = 'IMD SCORE'

imd_2010_london = pd.merge(
    imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL]],
    london_lookup,
    left_on=IMD_2010_LSOA_COL, right_on='lsoa11cd'
)
msoa_imd_2010 = (
    imd_2010_london
    .groupby(['msoa11cd', 'ladnm'])[IMD_2010_SCORE_COL]
    .mean().reset_index()
    .rename(columns={IMD_2010_SCORE_COL: 'IMD_2010'})
)

# ---- IMD 2019 → MSOA ----
imd_2019_london = pd.merge(
    imd_2019[['LSOA code (2011)', 'Index of Multiple Deprivation (IMD) Score']],
    london_lookup,
    left_on='LSOA code (2011)', right_on='lsoa11cd'
)
msoa_imd_2019 = (
    imd_2019_london
    .groupby('msoa11cd')['Index of Multiple Deprivation (IMD) Score']
    .mean().reset_index()
    .rename(columns={'Index of Multiple Deprivation (IMD) Score': 'IMD_2019'})
)

# ---- Combine ----
msoa_wealth = pd.merge(msoa_imd_2010, msoa_imd_2019, on='msoa11cd', how='inner')
msoa_wealth['IMD_Change'] = msoa_wealth['IMD_2019'] - msoa_wealth['IMD_2010']

print(f'London MSOAs with both IMD years: {len(msoa_wealth)}')
msoa_wealth.head()

---
## 4. Wealth Decile Assignment (Fixed 2010 Baseline)

Deciles are assigned using **IMD 2010 only**, so the classification is
identical regardless of which census year we analyse.

Convention: 1 = most deprived, 10 = least deprived (wealthiest).

In [ ]:
msoa_wealth['Wealth_Decile'] = pd.qcut(
    msoa_wealth['IMD_2010'], 10, labels=False
) + 1
msoa_wealth['Wealth_Decile'] = 11 - msoa_wealth['Wealth_Decile']

wealth_dict = msoa_wealth.set_index('msoa11cd')['Wealth_Decile'].to_dict()

decile_counts = msoa_wealth['Wealth_Decile'].value_counts().sort_index()
print(f'Average MSOAs per decile: {decile_counts.mean():.1f}')
print(decile_counts)

---
## 5. O-D Flow Processing & Cascade Feature Computation

Build the London-to-London flow table with decile assignments,
then compute per-MSOA cascade features.

In [ ]:
# ---- 5a. Clean O-D data & assign deciles ----
od = census_od_2021.copy()
od = od[od[ORIGIN_COL].astype(str) != '-8'].copy()

od['Origin_Decile'] = od['origin_msoa11'].map(wealth_dict)
od['Dest_Decile']   = od['dest_msoa11'].map(wealth_dict)

# Keep only London-to-London flows
london_flow = od.dropna(subset=['Origin_Decile', 'Dest_Decile']).copy()
london_flow['Origin_Decile'] = london_flow['Origin_Decile'].astype(int)
london_flow['Dest_Decile']   = london_flow['Dest_Decile'].astype(int)

# Detect count column
count_cols = [c for c in london_flow.columns if 'observation' in c.lower() or 'count' in c.lower()]
COUNT_COL = count_cols[0] if count_cols else '_count'
if COUNT_COL == '_count':
    london_flow[COUNT_COL] = 1

london_flow['Decile_Shift'] = london_flow['Dest_Decile'] - london_flow['Origin_Decile']

print(f'Total O-D records (raw):     {len(census_od_2021):,}')
print(f'After removing non-migrants: {len(od):,}')
print(f'London-to-London flows:      {len(london_flow):,}')
print(f'Total migrants:              {london_flow[COUNT_COL].sum():,.0f}')
print(f'Count column used:           {COUNT_COL}')

In [ ]:
# ---- 5b. Flow matrix (decile × decile) ----
flow_matrix = london_flow.pivot_table(
    index='Origin_Decile', columns='Dest_Decile',
    values=COUNT_COL, aggfunc='sum', fill_value=0
)

# ---- 5c. Aggregate flow direction summary ----
total_migrants = london_flow[COUNT_COL].sum()
upward   = london_flow[london_flow['Decile_Shift'] > 0][COUNT_COL].sum()
downward = london_flow[london_flow['Decile_Shift'] < 0][COUNT_COL].sum()
lateral  = london_flow[london_flow['Decile_Shift'] == 0][COUNT_COL].sum()

flow_direction = pd.Series({
    'Upward (to wealthier)': upward,
    'Downward (to more deprived)': downward,
    'Lateral (same decile)': lateral,
    'Total': total_migrants
})

# ---- 5d. Decile shift distribution ----
shift_dist = london_flow.groupby('Decile_Shift')[COUNT_COL].sum()

print('=== Flow Direction Summary (2021) ===')
for label, val in flow_direction.items():
    pct = val / total_migrants * 100 if label != 'Total' else 100
    print(f'  {label:35s} {val:>10,.0f}  ({pct:.1f}%)')

In [ ]:
# ---- 5e. Per-MSOA cascade features ----
msoa_analysis = msoa_wealth[['msoa11cd', 'ladnm', 'Wealth_Decile',
                              'IMD_2010', 'IMD_2019', 'IMD_Change']].copy()

# Inflow from wealthier areas (origin decile > dest decile)
inflow_w = (
    london_flow[london_flow['Origin_Decile'] > london_flow['Dest_Decile']]
    .groupby('dest_msoa11')[COUNT_COL].sum()
    .rename('Inflow_Wealthier')
)

# Outflow to more deprived areas (dest decile < origin decile)
outflow_p = (
    london_flow[london_flow['Dest_Decile'] < london_flow['Origin_Decile']]
    .groupby('origin_msoa11')[COUNT_COL].sum()
    .rename('Outflow_Poorer')
)

# Total inflow & outflow
total_in = london_flow.groupby('dest_msoa11')[COUNT_COL].sum().rename('Total_Inflow')
total_out = london_flow.groupby('origin_msoa11')[COUNT_COL].sum().rename('Total_Outflow')

# Merge all onto MSOA table
for s in [inflow_w, outflow_p, total_in, total_out]:
    msoa_analysis = msoa_analysis.merge(s, left_on='msoa11cd', right_index=True, how='left')

msoa_analysis = msoa_analysis.fillna(0)

# Derived features
msoa_analysis['Net_Cascade'] = msoa_analysis['Inflow_Wealthier'] - msoa_analysis['Outflow_Poorer']
msoa_analysis['Cascade_Ratio'] = np.where(
    msoa_analysis['Outflow_Poorer'] > 0,
    msoa_analysis['Inflow_Wealthier'] / msoa_analysis['Outflow_Poorer'],
    np.nan
)
msoa_analysis['Pct_Inflow_Wealthier'] = (
    msoa_analysis['Inflow_Wealthier'] / msoa_analysis['Total_Inflow'].replace(0, np.nan) * 100
)

print(f'MSOAs with flow data: {(msoa_analysis["Total_Inflow"] > 0).sum()}')
msoa_analysis.describe().round(1)

In [ ]:
# ---- 5f. Decile-level summary table ----
decile_summary = msoa_analysis.groupby('Wealth_Decile').agg(
    MSOA_Count=('msoa11cd', 'count'),
    Avg_Inflow_Wealthier=('Inflow_Wealthier', 'mean'),
    Avg_Outflow_Poorer=('Outflow_Poorer', 'mean'),
    Avg_Net_Cascade=('Net_Cascade', 'mean'),
    Avg_IMD_Change=('IMD_Change', 'mean'),
).round(1)
decile_summary['Avg_Cascade_Ratio'] = (
    msoa_analysis.groupby('Wealth_Decile')['Cascade_Ratio'].mean().round(2)
)

# ---- 5g. Borough-level summary table ----
borough_summary = msoa_analysis.groupby('ladnm').agg(
    Num_MSOAs=('msoa11cd', 'count'),
    Avg_IMD_2010=('IMD_2010', 'mean'),
    Avg_IMD_Change=('IMD_Change', 'mean'),
    Total_Net_Cascade=('Net_Cascade', 'sum'),
    Avg_Cascade_Ratio=('Cascade_Ratio', 'mean'),
    Total_Inflow=('Total_Inflow', 'sum'),
    Total_Outflow=('Total_Outflow', 'sum')
).round(2).sort_values('Total_Net_Cascade', ascending=False)

print('Data preprocessing complete.')
print(f'  msoa_wealth:    {msoa_wealth.shape}')
print(f'  london_flow:    {london_flow.shape}')
print(f'  msoa_analysis:  {msoa_analysis.shape}')
print(f'  flow_matrix:    {flow_matrix.shape}')
print(f'  decile_summary: {decile_summary.shape}')
print(f'  borough_summary:{borough_summary.shape}')

---
# PART B — ANALYSIS & VISUALISATION

All data tables are now ready. The sections below are purely
plots, statistical tests, and interpretation.

---

## 6. IMD Baseline Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(msoa_wealth['IMD_2010'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Mean IMD 2010 Score (higher = more deprived)')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Chart 1A: IMD 2010 Score Distribution (London MSOAs)')

dc = msoa_wealth['Wealth_Decile'].value_counts().sort_index()
axes[1].bar(dc.index, dc.values, color='teal', edgecolor='white')
axes[1].set_xlabel('Wealth Decile (1 = most deprived, 10 = wealthiest)')
axes[1].set_ylabel('Number of MSOAs')
axes[1].set_title('Chart 1B: MSOA Count per Wealth Decile (2010 Baseline)')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig1_imd_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

##### NOTE

Chart 1A: London's IMD distribution is right-skewed. Most MSOAs have low-to-moderate deprivation, with a long tail of highly deprived areas.

> compare with the whole of England

Chart 1B: The decile chart gives roughly equal counts (~98 MSOAs per decile).

> The number of MSOAs is identical between 2010 and 2019 because both IMD datasets use 2011 LSOA codes, and we aggregate to the same set of 2011 MSOAs. The scores change, but the geography doesn't.

---
## 7. IMD Change Analysis (Temporal Dimension)

Which MSOAs became less deprived between 2010 and 2019?
This serves as our independent validation of gentrification.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(msoa_wealth['IMD_Change'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('IMD Change (2019 − 2010)')
axes[0].set_ylabel('Number of MSOAs')
axes[0].set_title('Chart 2A: IMD Score Change across London MSOAs\n(negative = became less deprived)')

decile_change = msoa_wealth.groupby('Wealth_Decile')['IMD_Change'].mean()
colors = ['coral' if v < 0 else 'steelblue' for v in decile_change.values]
axes[1].bar(decile_change.index, decile_change.values, color=colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Wealth Decile (2010 baseline)')
axes[1].set_ylabel('Mean IMD Change')
axes[1].set_title('Chart 2B: Average IMD Change by 2010 Wealth Decile')
axes[1].set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig2_imd_change.png', dpi=150, bbox_inches='tight')
plt.show()

n_less = (msoa_wealth['IMD_Change'] < 0).sum()
n_more = (msoa_wealth['IMD_Change'] > 0).sum()
n_total = len(msoa_wealth)
print(f'MSOAs that became less deprived: {n_less} ({n_less/n_total*100:.1f}%)')
print(f'MSOAs that became more deprived: {n_more} ({n_more/n_total*100:.1f}%)')

##### NOTE

> change in rank is more reasonable — rank analysis

Chart 2A: ~83% of MSOAs became less deprived between 2010 and 2019. London as a whole saw widespread deprivation decline.

Chart 2B: the most deprived MSOAs (decile 1) saw the largest IMD drops (~11 points), while the wealthiest (decile 9–10) barely changed. Consistent with gentrification theory: the poorest areas are changing the fastest.

But IMD alone can't distinguish between genuine neighbourhood improvement and demographic displacement.

_Did the neighbourhood actually improve, or did the poor residents just get replaced by wealthier ones?_

---
## 8. Origin–Destination Flow Analysis (2021)

In [ ]:
# Flow matrix heatmap
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(
    flow_matrix, annot=True, fmt=',.0f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Number of migrants'}, ax=ax
)
ax.set_xlabel('Destination Wealth Decile')
ax.set_ylabel('Origin Wealth Decile')
ax.set_title('Migration Flows between Wealth Deciles\n(London, 2021 Census | Fixed IMD 2010 Deciles)')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig3_od_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

##### NOTE

- Largest flows are along the diagonal (same-decile moves) — people mostly move to areas of similar deprivation.
- Most deprived areas have highest internal churn (decile 1→1 is the single largest flow).
- Flows from deprived origins to wealthy destinations are much smaller.

> Questions: How to understand the churn from decile 10 to 1?

> Worth looking at the flow rate: divide by "population at risk" of moving

In [ ]:
# Decile shift distribution
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d73027' if x < 0 else '#4575b4' if x > 0 else '#999999' for x in shift_dist.index]
ax.bar(shift_dist.index, shift_dist.values, color=colors, edgecolor='white')
ax.set_xlabel('Decile Shift (negative = moved to more deprived area)')
ax.set_ylabel('Number of migrants')
ax.set_title('Distribution of Wealth-Decile Shifts in London Migration (2021)')
ax.set_xticks(range(shift_dist.index.min(), shift_dist.index.max() + 1))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig4_decile_shift.png', dpi=150, bbox_inches='tight')
plt.show()

##### NOTE

- Upward moves outnumber downward moves at nearly every magnitude (consistent with 40.2% vs 35.5% split).
- Need to know if downward moves are spatially concentrated in MSOAs that simultaneously received affluent inflows.

---
## 9. Cascade Features by Decile & Borough

In [ ]:
print('=== Average Features by Wealth Decile ===')
display(decile_summary)

In [ ]:
# Monotonicity tests
r, p = stats.spearmanr(decile_summary.index, decile_summary['Avg_Net_Cascade'])
print(f'Wealth Decile vs Net Cascade:  rho = {r:+.3f}, p = {p:.6f}')

r, p = stats.spearmanr(decile_summary['Avg_Net_Cascade'], decile_summary['Avg_IMD_Change'])
print(f'Net Cascade vs IMD Change:     rho = {r:+.3f}, p = {p:.6f}')

##### NOTE

- Around decile 5–6, `Avg_Net_Cascade` flips from positive to negative (crossover point).
  - MSOAs in deciles 1–5 are net receivers of wealthier inflows.
  - MSOAs in deciles 6–10 are net senders.
- `Avg_Cascade_Ratio` drops monotonically from ~8.0 (decile 2) to ~0.3 (decile 9).
- Spearman rho close to −1.0 with p < 0.05 confirms strong monotonic relationships.

> Question: which metric (Net_Cascade or Cascade_Ratio) is better for the index?

In [ ]:
# Top 15 MSOAs by Net Cascade
top = msoa_analysis.nlargest(15, 'Net_Cascade')

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top['ladnm'], top['Net_Cascade'], color='coral', edgecolor='white')
ax.set_xlabel('Net Cascade (Inflow from wealthier − Outflow to poorer)')
ax.set_title('Top 15 MSOAs by Gentrification Pressure')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig5_top_cascade.png', dpi=150, bbox_inches='tight')
plt.show()

##### NOTE

- Strongest gentrification pressure in Tower Hamlets, Islington, Greenwich, and Hackney — all widely recognised as gentrifying in the literature.
- Good face validity for the index.

---
## 10. Validation: Cascade Index vs IMD Change

**Key test**: MSOAs with high cascade pressure should independently show falling IMD scores.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 6A: Net Cascade vs IMD Change
ax = axes[0]
sc = ax.scatter(
    msoa_analysis['Net_Cascade'], msoa_analysis['IMD_Change'],
    c=msoa_analysis['Wealth_Decile'], cmap='RdYlGn',
    s=25, alpha=0.6, edgecolors='grey', linewidth=0.3
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Net Cascade (2021 flows)')
ax.set_ylabel('IMD Change (2010 → 2019)')
ax.set_title('Plot 6A: Cascade Index vs Deprivation Change')

# Plot 6B: IMD baseline vs IMD Change, coloured by cascade
ax = axes[1]
sc2 = ax.scatter(
    msoa_analysis['IMD_2010'], msoa_analysis['IMD_Change'],
    c=msoa_analysis['Net_Cascade'], cmap='RdBu_r',
    s=25, alpha=0.6, edgecolors='grey', linewidth=0.3
)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('IMD 2010 Score (higher = more deprived)')
ax.set_ylabel('IMD Change (2010 → 2019)')
ax.set_title('Plot 6B: Baseline Deprivation vs Change\n(coloured by Net Cascade)')
plt.colorbar(sc2, label='Net Cascade', ax=ax)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig6_validation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation statistics
print('=== Validation Correlations ===')
for col, label in [('Net_Cascade', 'Net Cascade'),
                    ('Cascade_Ratio', 'Cascade Ratio'),
                    ('Pct_Inflow_Wealthier', '% Inflow from Wealthier')]:
    valid = msoa_analysis[[col, 'IMD_Change']].dropna()
    r, p = stats.pearsonr(valid[col], valid['IMD_Change'])
    print(f'{label:30s} vs IMD Change:  r = {r:+.3f},  p = {p:.4f}')

##### NOTE

- Net Cascade vs IMD Change (r ≈ −0.42): moderate-to-strong negative correlation — more cascade pressure → bigger deprivation drops.
- Cascade Ratio not significant — ratio is unstable with very low outflows.
- Absolute volume of cascade flows matters more than ratios for predicting changes.

---
## 11. Borough-Level Summary

In [ ]:
display(borough_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
colors = ['coral' if v > 0 else 'steelblue' for v in borough_summary['Total_Net_Cascade']]
ax.barh(borough_summary.index, borough_summary['Total_Net_Cascade'],
        color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Total Net Cascade')
ax.set_title('Net Cascade by London Borough (2021)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig7_borough_cascade.png', dpi=150, bbox_inches='tight')
plt.show()

##### NOTE

- Top cascade boroughs: Tower Hamlets, Hackney, Lambeth, Newham, Islington — all historically deprived inner-London areas with well-documented gentrification.
- Negative cascade boroughs: Barnet, Redbridge, Merton, Richmond — wealthier outer-London suburbs.
- Outer boroughs may be receiving displaced residents from gentrifying inner boroughs.

> map the result

---
## 12. Export

In [ ]:
msoa_analysis.to_csv(OUTPUT_DIR / 'msoa_cascade_features.csv', index=False)
borough_summary.to_csv(OUTPUT_DIR / 'borough_summary.csv')
flow_matrix.to_csv(OUTPUT_DIR / 'flow_matrix_decile.csv')

print('Exported to outputs/:')
print('  - msoa_cascade_features.csv')
print('  - borough_summary.csv')
print('  - flow_matrix_decile.csv')

---
## Discussion

1. **Option C methodology**: Fixed 2010 IMD baseline for cascade classification,
   with IMD change (2010→2019) as independent validation.

2. **2011 O-D data limitation**: MSOA-level residential migration from the 2011
   Census is safeguarded (not publicly downloadable). Options: apply for WICID
   access, or use LA-level 2011 data as a supplementary analysis.

3. **Cascade Index formalisation**: Current metric is
   `Net_Cascade = Inflow_from_wealthier − Outflow_to_poorer`.
   Should we normalise by total flow volume? Weight by decile shift magnitude?

4. **COVID caveat**: The 2021 Census (21 March 2021) was during COVID restrictions.
   Migration patterns in the preceding year may be atypical.

5. **Geography**: ~2.5% of MSOAs nationally changed between 2011 and 2021.
   We excluded these. How many are in London, and does it matter?

6. **Next steps**: Alluvial plots, spatial mapping with geopandas,
   and potentially ODMG04EW (migration by NS-SEC) for individual-level
   socio-economic classification rather than area-based IMD proxy.

7. Rank analysis

8. Paper for gentrification metrics

9. Population at risk analysis & O-D churn / inflow (outflow) population

10. Sankey plots and London map

11. Displacement project exploration
    — spatially, per MSOA, top 10 inflow MSOA & top 10 outflow MSOA
    — within London and UK

12. Take centroid of each MSOA, calculate avg distance of flows

13. Start to write findings as stories

14. 2011 Census OD data (see eda-v3.ipynb)